<a href="https://colab.research.google.com/github/TimofeyProtasov/diploma/blob/main/days/work_2304%2B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q triton trl evaluate ninja

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 46.5 MB/s eta 0:00:00


In [2]:
import re
import torch
import random
import numpy as np
import pandas as pd
import time
import evaluate
from typing import Optional
from sklearn.model_selection import train_test_split
from dataclasses import dataclass, field
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets
from trl import SFTTrainer, SFTConfig
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    PeftConfig
)
from tqdm import tqdm
from functools import wraps
from torch.utils.data import DataLoader

In [3]:
def measure_metrics(metric_name: str):
    """
    Декоратор для измерения времени выполнения и пиковой памяти GPU метода.
    Сохраняет результаты в self.metrics с ключами:
    - f"{metric_name}_time_min"
    - f"{metric_name}_peak_memory_gb"
    """
    def decorator(func):
        @wraps(func)
        def wrapper(self, *args, **kwargs):
            # Сбрасываем статистику пиковой памяти, если доступна CUDA
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()

            start_time = time.time()

            # Выполняем сам метод
            result = func(self, *args, **kwargs)

            elapsed_min = (time.time() - start_time) / 60.0
            self.metrics[f"{metric_name}_time_min"] = round(elapsed_min, 2)

            if torch.cuda.is_available():
                peak_memory_gb = torch.cuda.max_memory_allocated() / (1024**3)
                self.metrics[f"{metric_name}_peak_memory_gb"] = round(peak_memory_gb, 2)
            else:
                self.metrics[f"{metric_name}_peak_memory_gb"] = 0.0

            return result
        return wrapper
    return decorator

In [4]:
@dataclass
class RAGExperiment:
    dataset_name: str = 'phatvo/hotpotqa-raft'
    oracle_presence_ratio: float = 1.0
    train_size: int = 10
    test_size: int = 2
    model_name: str = 'Qwen/Qwen2.5-0.5B-Instruct'
    peft_config: Optional[PeftConfig] = field(default_factory=lambda: LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    ))
    eval_epochs: list = field(default_factory=lambda: [3])
    per_device_train_batch_size: int = 4
    gradient_accumulation_steps: int = 2
    learning_rate: float = 5e-5
    warmup_steps: int = 5
    seed: int = 1

    def __post_init__(self):
        random.seed(self.seed)
        self.squad_metric = evaluate.load("squad")
        self.metrics = {}

        # Метрики качества
        self.f1_by_epoch = {}
        self.perplexity_by_epoch = {}

        # Время и память обучения
        self.train_time_by_epoch = {}
        self.train_memory_by_epoch = {}

        # Время и память оценки F1
        self.f1_time_by_epoch = {}
        self.f1_memory_by_epoch = {}

        # Время и память оценки перплексии
        self.perplexity_time_by_epoch = {}
        self.perplexity_memory_by_epoch = {}

    def prepare_data(self):
        dataset: DatasetDict = load_dataset('phatvo/hotpotqa-raft')


        def presence_oracle(example):
            return example['oracle_context'] in example['instruction']


        dataset_with_oracle: Dataset = dataset['train'].filter(lambda ex: presence_oracle(ex))
        dataset_without_oracle: Dataset = dataset['train'].filter(lambda ex: not presence_oracle(ex))

        self.sample_size = self.train_size + self.test_size

        num_with: int = round(self.sample_size * self.oracle_presence_ratio)
        num_without: int = round(self.sample_size * (1 - self.oracle_presence_ratio))

        if (num_with > len(dataset_with_oracle)) or (num_without > len(dataset_without_oracle)):
            raise ValueError("not enough samples in dataset")

        indices_with: list = random.sample(range(len(dataset_with_oracle)), num_with)
        size_train_with = round(self.train_size * self.oracle_presence_ratio)

        indices_without: list = random.sample(range(len(dataset_without_oracle)), num_without)
        size_train_without = round(self.train_size * (1 - self.oracle_presence_ratio))

        idx_train_with_oracle = indices_with[:size_train_with]
        idx_test_with_oracle = indices_with[size_train_with:num_with]

        idx_train_without_oracle = indices_without[:size_train_without]
        idx_test_without_oracle = indices_without[size_train_without:num_without]

        def get_shuffle_dataset(ids_with, ids_without):
            parts = []
            if ids_with:
                parts.append(dataset_with_oracle.select(ids_with))
            if ids_without:
                parts.append(dataset_without_oracle.select(ids_without))
            if parts:
                return concatenate_datasets(parts).shuffle(seed=self.seed)
            else:
                raise ValueError('empty sample')


        self.train_dataset: Dataset = get_shuffle_dataset(idx_train_with_oracle, idx_train_without_oracle)
        self.test_dataset: Dataset = get_shuffle_dataset(idx_test_with_oracle, idx_test_without_oracle)


    def prepare_model(self):
        attn = "eager" if isinstance(self.peft_config, PrefixTuningConfig) else "sdpa"
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            dtype=torch.bfloat16,
            device_map="auto",
            attn_implementation=attn,
        )

    def prepare_tokenizer(self):
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_name
        )
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token


    def format_data(self, dataset):
        """Форматируем данные в prompt/completion для SFTTrainer."""
        def format_sft_example(example):
            prompt_messages = [{"role": "user", "content": example["instruction"]}]
            prompt = self.tokenizer.apply_chat_template(
                prompt_messages,
                tokenize=False,
                add_generation_prompt=True
            )
            completion = example["cot_answer"]
            return {"prompt": prompt, "completion": completion}

        return dataset.map(
            format_sft_example,
            remove_columns=dataset.column_names
        )


    def add_peft_if_exist(self):
        self.model = get_peft_model(self.model, self.peft_config) if self.peft_config else self.model
        self.metrics["trainable_params_m"] = self.model.num_parameters(only_trainable=True) / 1e6


    def prepare_training(self):
        total_epochs = self.eval_epochs[-1]   # последняя эпоха в списке

        peft_name = type(self.peft_config).__name__.replace("Config", "")
        output_dir_ = f"./qwen-raft-{peft_name}-train{self.train_size}"

        self.training_args = SFTConfig(
            output_dir=output_dir_,
            num_train_epochs=total_epochs,
            per_device_train_batch_size=self.per_device_train_batch_size,
            gradient_accumulation_steps=self.gradient_accumulation_steps,
            learning_rate=self.learning_rate,
            warmup_steps=self.warmup_steps,
            logging_steps=10,
            save_strategy="epoch",           # сохраняем после каждой эпохи
            save_steps=None,                 # отключаем шаговое сохранение
            save_total_limit=2,
            bf16=True,
            report_to="none",
            remove_unused_columns=False,

            max_length=1280,
            packing=False,
            completion_only_loss=True,
        )

        self.trainer = SFTTrainer(
            model=self.model,
            args=self.training_args,
            train_dataset=self.formatted_train_dataset,
            processing_class=self.tokenizer,
        )

    @measure_metrics("train")
    def train(self, resume: bool):
        self.trainer.train(resume_from_checkpoint=resume)


    def extract_answer(self, text: str) -> Optional[str]:
        """
        Извлекает ответ после <ANSWER>:.
        Поддерживает многострочные ответы до конца текста или до следующего маркера.
        """
        match = re.search(r"<ANSWER>:\s*(.*?)(?=\n<|$)", text, re.DOTALL)
        if match:
            return match.group(1).strip()
        return None


    @measure_metrics("evaluate_f1")
    def evaluate_f1(self, batch_size: int = 8):
        original_padding_side = self.tokenizer.padding_side
        self.tokenizer.padding_side = "left"

        def collate_fn(batch):
            prompts = [item["prompt"] for item in batch]
            completions = [item["completion"] for item in batch]
            true_answers = [self.extract_answer(c) for c in completions]

            tokenized = self.tokenizer(
                prompts,
                padding=True,
                truncation=True,
                max_length=self.training_args.max_length,
                return_tensors="pt"
            )
            return {
                "prompts": prompts,
                "input_ids": tokenized["input_ids"],
                "attention_mask": tokenized["attention_mask"],
                "true_answers": true_answers,
                "ids": [str(i) for i in range(len(batch))]
            }

        dataloader = DataLoader(
            self.formatted_test_dataset,
            batch_size=batch_size,
            collate_fn=collate_fn,
            shuffle=False
        )

        f1_scores = []

        for batch in tqdm(dataloader, desc=f"Evaluating F1 (batch_size={batch_size})"):
            input_ids = batch["input_ids"].to(self.model.device)
            attention_mask = batch["attention_mask"].to(self.model.device)

            with torch.no_grad():
                outputs = self.model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=1024,
                    do_sample=False,
                    pad_token_id=self.tokenizer.eos_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                    temperature=None,
                    top_p=None,
                    top_k=None,
                )

            for i, (prompt, true_answer, example_id) in enumerate(zip(
                batch["prompts"], batch["true_answers"], batch["ids"]
            )):
                if true_answer is None:
                    continue

                prompt_len = attention_mask[i].sum().item()  # реальная длина промпта
                generated_ids = outputs[i][prompt_len:]
                generated_part = self.tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
                pred_answer = self.extract_answer(generated_part)

                if pred_answer is None:
                    f1 = 0.0
                else:
                    pred_dict = {"prediction_text": pred_answer, "id": example_id}
                    ref_dict = {"answers": {"answer_start": [0], "text": [true_answer]}, "id": example_id}
                    result = self.squad_metric.compute(predictions=[pred_dict], references=[ref_dict])
                    f1 = result["f1"] / 100.0

                f1_scores.append(f1)

        self.tokenizer.padding_side = original_padding_side

        avg_f1 = np.mean(f1_scores) if f1_scores else 0.0
        self.metrics["f1"] = round(float(avg_f1), 2)


    @measure_metrics("evaluate_perplexity")
    def evaluate_perplexity(self):
        """
        Вычисляет перплексию на тестовом наборе без батчинга.
        Потери считаются только по completion.
        """
        total_loss = 0.0
        total_tokens = 0

        for example in tqdm(self.formatted_test_dataset, desc="Calculating perplexity"):
            prompt = example["prompt"]
            completion = example["completion"]
            full_text = prompt + completion

            # Токенизируем полный текст (с параметрами обучения)
            tokenized_full = self.tokenizer(
                full_text,
                truncation=True,
                max_length=self.training_args.max_length,
                return_tensors="pt"
            )
            input_ids = tokenized_full["input_ids"][0]

            # Токенизируем ТОЛЬКО промпт с теми же параметрами, чтобы узнать его длину
            tokenized_prompt = self.tokenizer(
                prompt,
                truncation=True,
                max_length=self.training_args.max_length,
                return_tensors="pt"
            )
            prompt_len = tokenized_prompt["input_ids"].shape[1]

            # Если полный текст был обрезан и промпт занял всё окно,
            # то completion мог не попасть вообще. В таком случае пропускаем пример.
            if prompt_len >= len(input_ids):
                continue

            # Создаём labels: копируем input_ids, маскируем промпт
            labels = input_ids.clone()
            labels[:prompt_len] = -100

            # Переносим на устройство
            input_ids = input_ids.unsqueeze(0).to(self.model.device)
            labels = labels.unsqueeze(0).to(self.model.device)
            attention_mask = torch.ones_like(input_ids).to(self.model.device)

            with torch.no_grad():
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss

            num_tokens = (labels != -100).sum().item()
            if num_tokens > 0:
                total_loss += loss.item() * num_tokens
                total_tokens += num_tokens

        avg_loss = total_loss / total_tokens if total_tokens > 0 else float("inf")
        perplexity = np.exp(avg_loss)

        self.metrics["perplexity"] = round(float(perplexity), 2)


    def run(self):
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        self.prepare_data()
        self.prepare_model()
        self.prepare_tokenizer()
        self.formatted_train_dataset = self.format_data(self.train_dataset)
        self.formatted_test_dataset = self.format_data(self.test_dataset)
        self.add_peft_if_exist()
        self.prepare_training()
        self.metrics["before_train_peak_memory_gb"] = round(torch.cuda.max_memory_allocated() / (1024**3), 2)

        current_epoch = 0
        for target_epoch in self.eval_epochs:
            if target_epoch <= current_epoch:
                continue

            self.training_args.num_train_epochs = target_epoch
            resume = (current_epoch > 0)

            # Обучение
            self.train(resume)
            current_epoch = target_epoch

            self.train_time_by_epoch[current_epoch] = self.metrics.get("train_time_min", 0.0)
            self.train_memory_by_epoch[current_epoch] = self.metrics.get("train_peak_memory_gb", 0.0)

            self.model.eval()

            # Оценка F1
            self.evaluate_f1(batch_size=8)
            self.f1_by_epoch[current_epoch] = self.metrics["f1"]
            self.f1_time_by_epoch[current_epoch] = self.metrics.get("evaluate_f1_time_min", 0.0)
            self.f1_memory_by_epoch[current_epoch] = self.metrics.get("evaluate_f1_peak_memory_gb", 0.0)

            # Оценка перплексии
            self.evaluate_perplexity()
            self.perplexity_by_epoch[current_epoch] = self.metrics["perplexity"]
            self.perplexity_time_by_epoch[current_epoch] = self.metrics.get("evaluate_perplexity_time_min", 0.0)
            self.perplexity_memory_by_epoch[current_epoch] = self.metrics.get("evaluate_perplexity_peak_memory_gb", 0.0)

        # Собираем всё в общий словарь метрик
        self.metrics["f1_by_epoch"] = self.f1_by_epoch
        self.metrics["perplexity_by_epoch"] = self.perplexity_by_epoch
        self.metrics["train_time_by_epoch"] = self.train_time_by_epoch
        self.metrics["train_memory_by_epoch"] = self.train_memory_by_epoch
        self.metrics["f1_time_by_epoch"] = self.f1_time_by_epoch
        self.metrics["f1_memory_by_epoch"] = self.f1_memory_by_epoch
        self.metrics["perplexity_time_by_epoch"] = self.perplexity_time_by_epoch
        self.metrics["perplexity_memory_by_epoch"] = self.perplexity_memory_by_epoch

In [5]:
from peft import (
    LoraConfig,
    BOFTConfig,
    PrefixTuningConfig,
    IA3Config,
    TaskType
)

# 1. LoRA — базовый (низкоранговая перепараметризация)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# 2. BOFT — ортогональная тонкая настройка (принципиально иной подход)
boft_config = BOFTConfig(
    boft_block_size=4,
    boft_n_butterfly_factor=1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    boft_dropout=0.0,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# 3. Prefix Tuning — мягкие промпты ко всем слоям
prefix_config = PrefixTuningConfig(
    num_virtual_tokens=10,
    encoder_hidden_size=896,       # размерность скрытого слоя Qwen2.5‑0.5B
    prefix_projection=True,        # стабилизирует обучение (P‑Tuning v2)
    task_type=TaskType.CAUSAL_LM,
)

# 4. IA³ — масштабирование ключей, значений и FFN
ia3_config = IA3Config(
    target_modules=["k_proj", "v_proj", "down_proj"],
    feedforward_modules=["down_proj"],
    task_type=TaskType.CAUSAL_LM,
)

In [6]:
import os
import pandas as pd
from pathlib import Path
import torch
import gc

# Монтируем Google Drive (если ещё не)
from google.colab import drive
drive.mount('/content/drive')

# Путь для сохранения CSV
SAVE_DIR = "/content/drive/MyDrive/rag_experiments_second"
os.makedirs(SAVE_DIR, exist_ok=True)

# Параметры экспериментов
train_sizes = [100, 300, 500, 700, 900, 1100]
test_size = 100
eval_epochs = [4, 6, 8, 10, 12, 14]

def flatten_metrics(metrics: dict, train_size: int) -> dict:
    """Преобразует вложенные словари метрик в плоский словарь для одной строки CSV."""
    flat = {"train_size": train_size}
    for key, value in metrics.items():
        if isinstance(value, dict):
            # Например, "f1_by_epoch": {5: 0.5, 10: 0.6, ...}
            for epoch, val in value.items():
                # Убираем суффикс "_by_epoch" и добавляем номер эпохи
                base_key = key.replace("_by_epoch", "")
                flat[f"{base_key}_epoch{epoch}"] = val
        else:
            flat[key] = value
    return flat


Mounted at /content/drive


In [ ]:
# Основной цикл
for train_size in train_sizes:
    csv_path = os.path.join(SAVE_DIR, f"results_train_{train_size}_prefix_config.csv")

    # Проверяем, не сохранён ли уже результат
    if os.path.exists(csv_path):
        print(f"Результаты для train_size={train_size} уже существуют, пропускаем.")
        continue

    print(f"\n{'='*50}")
    print(f"Запуск эксперимента для train_size={train_size}")
    print(f"{'='*50}")

    try:
        # Создаём и запускаем эксперимент
        exp = RAGExperiment(
            train_size=train_size,
            test_size=test_size,
            eval_epochs=eval_epochs,
            peft_config=prefix_config
        )
        exp.run()

        # Формируем плоский словарь и сохраняем в CSV
        flat_metrics = flatten_metrics(exp.metrics, train_size)
        df = pd.DataFrame([flat_metrics])
        df.to_csv(csv_path, index=False)
        print(f"Результаты сохранены в {csv_path}")

    except Exception as e:
        print(f"Ошибка при обработке train_size={train_size}: {e}")
        # При желании можно сохранить информацию об ошибке и продолжить
        continue

    finally:
        # Освобождаем ресурсы
        del exp
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

In [7]:
# Основной цикл
for train_size in train_sizes:
    csv_path = os.path.join(SAVE_DIR, f"results_train_{train_size}_boft_config.csv")

    # Проверяем, не сохранён ли уже результат
    if os.path.exists(csv_path):
        print(f"Результаты для train_size={train_size} уже существуют, пропускаем.")
        continue

    print(f"\n{'='*50}")
    print(f"Запуск эксперимента для train_size={train_size}")
    print(f"{'='*50}")

    try:
        # Создаём и запускаем эксперимент
        exp = RAGExperiment(
            train_size=train_size,
            test_size=test_size,
            eval_epochs=eval_epochs,
            peft_config=boft_config
        )
        exp.run()

        # Формируем плоский словарь и сохраняем в CSV
        flat_metrics = flatten_metrics(exp.metrics, train_size)
        df = pd.DataFrame([flat_metrics])
        df.to_csv(csv_path, index=False)
        print(f"Результаты сохранены в {csv_path}")

    except Exception as e:
        print(f"Ошибка при обработке train_size={train_size}: {e}")
        # При желании можно сохранить информацию об ошибке и продолжить
        continue

    finally:
        # Освобождаем ресурсы
        del exp
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()


Запуск эксперимента для train_size=100


README.md:   0%|          | 0.00/651 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/6.30M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1855 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1855 [00:00<?, ? examples/s]

...

Tokenizing train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,1.137304
20,1.036886
30,0.949959
40,0.894606
50,0.894063


Evaluating F1 (batch_size=8):  15%|█▌        | 2/13 [06:23<35:08, 191.65s/it]


KeyboardInterrupt: 

In [9]:
# Основной цикл
for train_size in train_sizes:
    csv_path = os.path.join(SAVE_DIR, f"results_train_{train_size}_ia3_config.csv")

    # Проверяем, не сохранён ли уже результат
    if os.path.exists(csv_path):
        print(f"Результаты для train_size={train_size} уже существуют, пропускаем.")
        continue

    print(f"\n{'='*50}")
    print(f"Запуск эксперимента для train_size={train_size}")
    print(f"{'='*50}")

    try:
        # Создаём и запускаем эксперимент
        exp = RAGExperiment(
            train_size=train_size,
            test_size=test_size,
            eval_epochs=eval_epochs,
            peft_config=ia3_config
        )
        exp.run()

        # Формируем плоский словарь и сохраняем в CSV
        flat_metrics = flatten_metrics(exp.metrics, train_size)
        df = pd.DataFrame([flat_metrics])
        df.to_csv(csv_path, index=False)
        print(f"Результаты сохранены в {csv_path}")

    except Exception as e:
        print(f"Ошибка при обработке train_size={train_size}: {e}")
        # При желании можно сохранить информацию об ошибке и продолжить
        continue

    finally:
        # Освобождаем ресурсы
        del exp
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()


Запуск эксперимента для train_size=100


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

...

Step,Training Loss
1390,0.756156
1400,0.785779
1410,0.767696
1420,0.768336
1430,0.767713
1440,0.789316
1450,0.772560
1460,0.769102
1470,0.772038
1480,0.759954


Calculating perplexity: 100%|██████████| 100/100 [00:06<00:00, 16.49it/s]


Step,Training Loss
1660,0.740695
1670,0.749242
1680,0.755988
1690,0.745562
1700,0.758750
1710,0.781760
1720,0.754811
1730,0.761397
1740,0.744549
1750,0.753395


Calculating perplexity: 100%|██████████| 100/100 [00:05<00:00, 16.72it/s]


Результаты сохранены в /content/drive/MyDrive/rag_experiments_second/results_train_1100_ia3_config.csv


In [10]:
# Основной цикл
for train_size in train_sizes:
    csv_path = os.path.join(SAVE_DIR, f"results_train_{train_size}_lora_config.csv")

    # Проверяем, не сохранён ли уже результат
    if os.path.exists(csv_path):
        print(f"Результаты для train_size={train_size} уже существуют, пропускаем.")
        continue

    print(f"\n{'='*50}")
    print(f"Запуск эксперимента для train_size={train_size}")
    print(f"{'='*50}")

    try:
        # Создаём и запускаем эксперимент
        exp = RAGExperiment(
            train_size=train_size,
            test_size=test_size,
            eval_epochs=eval_epochs,
            peft_config=lora_config
        )
        exp.run()

        # Формируем плоский словарь и сохраняем в CSV
        flat_metrics = flatten_metrics(exp.metrics, train_size)
        df = pd.DataFrame([flat_metrics])
        df.to_csv(csv_path, index=False)
        print(f"Результаты сохранены в {csv_path}")

    except Exception as e:
        print(f"Ошибка при обработке train_size={train_size}: {e}")
        # При желании можно сохранить информацию об ошибке и продолжить
        continue

    finally:
        # Освобождаем ресурсы
        del exp
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()


Запуск эксперимента для train_size=100


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

...

Step,Training Loss
1360,0.268137
1370,0.234697
1380,0.231781
1390,0.230893
1400,0.239125
1410,0.257438
1420,0.253318
1430,0.236252
1440,0.239857
1450,0.228746


Calculating perplexity: 100%|██████████| 100/100 [00:09<00:00, 10.47it/s]


Результаты сохранены в /content/drive/MyDrive/rag_experiments_second/results_train_900_lora_config.csv

Запуск эксперимента для train_size=1100


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [11]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

In [ ]:
def load_all_results(save_dir: str) -> pd.DataFrame:
    all_files = Path(save_dir).glob("results_train_*.csv")
    df_list = []
    for file in all_files:
        df = pd.read_csv(file)
        df_list.append(df)
    if df_list:
        return pd.concat(df_list, ignore_index=True)
    else:
        return pd.DataFrame()

df_all = load_all_results(SAVE_DIR)
print(df_all)

In [12]:
train_sizes = [100, 300, 500, 700]
# Путь для сохранения CSV
SAVE_DIR = "/content/drive/MyDrive/rag_experiments_second_oracle"
os.makedirs(SAVE_DIR, exist_ok=True)

In [13]:
# Основной цикл
for train_size in train_sizes:
    csv_path = os.path.join(SAVE_DIR, f"results_train_{train_size}_ia3_config_oracle.csv")

    # Проверяем, не сохранён ли уже результат
    if os.path.exists(csv_path):
        print(f"Результаты для train_size={train_size} уже существуют, пропускаем.")
        continue

    print(f"\n{'='*50}")
    print(f"Запуск эксперимента для train_size={train_size}")
    print(f"{'='*50}")

    try:
        # Создаём и запускаем эксперимент
        exp = RAGExperiment(
            train_size=train_size,
            test_size=test_size,
            eval_epochs=eval_epochs,
            peft_config=ia3_config,
            oracle_presence_ratio=0.8
        )
        exp.run()

        # Формируем плоский словарь и сохраняем в CSV
        flat_metrics = flatten_metrics(exp.metrics, train_size)
        df = pd.DataFrame([flat_metrics])
        df.to_csv(csv_path, index=False)
        print(f"Результаты сохранены в {csv_path}")

    except Exception as e:
        print(f"Ошибка при обработке train_size={train_size}: {e}")
        # При желании можно сохранить информацию об ошибке и продолжить
        continue

    finally:
        # Освобождаем ресурсы
        del exp
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()


Запуск эксперимента для train_size=100


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

...

Step,Training Loss
890,0.991193
900,0.929052
910,0.932416
920,0.966918
930,0.994329
940,0.892235
950,0.997037
960,0.948036
970,0.965672
980,0.915846


Calculating perplexity: 100%|██████████| 100/100 [00:05<00:00, 16.73it/s]


Step,Training Loss
1060,0.877698
1070,0.967275
1080,0.965363
1090,0.949337
1100,0.941370
1110,0.908808
1120,0.970390
1130,0.909470
1140,0.959793
1150,0.968964


Calculating perplexity: 100%|██████████| 100/100 [00:06<00:00, 16.22it/s]


Результаты сохранены в /content/drive/MyDrive/rag_experiments_second_oracle/results_train_700_ia3_config_oracle.csv


In [14]:
# Основной цикл
for train_size in train_sizes:
    csv_path = os.path.join(SAVE_DIR, f"results_train_{train_size}_lora_config_oracle.csv")

    # Проверяем, не сохранён ли уже результат
    if os.path.exists(csv_path):
        print(f"Результаты для train_size={train_size} уже существуют, пропускаем.")
        continue

    print(f"\n{'='*50}")
    print(f"Запуск эксперимента для train_size={train_size}")
    print(f"{'='*50}")

    try:
        # Создаём и запускаем эксперимент
        exp = RAGExperiment(
            train_size=train_size,
            test_size=test_size,
            eval_epochs=eval_epochs,
            peft_config=lora_config,
            oracle_presence_ratio=0.8
        )
        exp.run()

        # Формируем плоский словарь и сохраняем в CSV
        flat_metrics = flatten_metrics(exp.metrics, train_size)
        df = pd.DataFrame([flat_metrics])
        df.to_csv(csv_path, index=False)
        print(f"Результаты сохранены в {csv_path}")

    except Exception as e:
        print(f"Ошибка при обработке train_size={train_size}: {e}")
        # При желании можно сохранить информацию об ошибке и продолжить
        continue

    finally:
        # Освобождаем ресурсы
        del exp
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()


Запуск эксперимента для train_size=100


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

...

Step,Training Loss
890,0.366813
900,0.321812
910,0.334105
920,0.342390
930,0.390257
940,0.297684
950,0.379211
960,0.346493
970,0.374003
980,0.313720


Calculating perplexity: 100%|██████████| 100/100 [00:09<00:00, 10.42it/s]


Step,Training Loss
1060,0.295561
1070,0.370052
1080,0.349411
1090,0.342787
1100,0.342363
1110,0.302589
1120,0.360816
1130,0.284596
1140,0.354853
1150,0.361813


Calculating perplexity: 100%|██████████| 100/100 [00:09<00:00, 10.38it/s]


Результаты сохранены в /content/drive/MyDrive/rag_experiments_second_oracle/results_train_700_lora_config_oracle.csv
